In [ ]:
import numpy as np
import pandas as pd
import os,sys,glob
%load_ext autoreload
%autoreload 2
import plotting_helpers as helper
import nibabel as nib
import scipy
import nilearn
from scipy.spatial.distance import cdist, pdist, squareform
import stats_helpers
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from nilearn.maskers import NiftiMasker
from nilearn.mass_univariate import permuted_ols
import hbn_config as hc
import hbn_utils as hu
import partlycloudy_utils as pcu
import partlycloudy_config as pcc
import infant_restmovie_utils as iru
from obspy.imaging.cm import viridis_white
import infant_restmovie_config as irc
from scipy import stats
from scipy.stats import wilcoxon
import parcelwise_regressions as pwr


In [ ]:
# Scrape together the CSV files
# fns = glob.glob(f'{hu.get_results_dir()}/*average_results_atlas_aggregated.csv')
# dataframes = []
# for f in fns:
#     a = pd.read_csv(f, index_col=0)
#     if 'ISC' in f:
#         a['measure'] = 'ISC'
#         a['score'] = a['score']
#     elif 'TPHATE_DiffOp_IDE' in f:
#         a['measure'] = 'TPHATE_DiffOp'
#         a['score'] = a['id_estimate']
#     else:# 'PCA' in f:
#         a['measure'] = 'PCA'
#         a['score'] = a['id_estimate']
#     a = a[["region_name","task","subject","AgeGroup","Age","measure",'score']]
#     dataframes.append(a)
# results_df = pd.concat(dataframes)
# results_df.reset_index(inplace=True, drop=True)
# results_df.to_csv(f'{hu.get_results_dir()}/parcelwise_results_ISC_IDE.csv')
results_df = pd.read_csv(f'{hu.get_results_dir()}/parcelwise_results_ISC_IDE.csv', index_col=0)

In [ ]:
temp = results_df.groupby(['subject', 'task', 'measure','AgeGroup']).mean(numeric_only=True).reset_index() 
temp = temp[temp['measure']=='TPHATE_DiffOp']

In [ ]:
temp.groupby(['task','AgeGroup']).count()

In [ ]:
model = smf.ols('score ~ AgeGroup*C(task)', data=temp).fit() 
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table )
# Extract the degrees of freedom and format for reporting
print("ANOVA Results Summary:")
print("=" * 70)
for effect in anova_table.index[:-1]:  # exclude 'Residual'
    f_stat = anova_table.loc[effect, 'F']
    df_effect = int(anova_table.loc[effect, 'df'])
    df_resid = int(anova_table.loc['Residual', 'df'])
    p_val = anova_table.loc[effect, 'PR(>F)']
    
    print(f"{effect}: F({df_effect}, {df_resid}) = {f_stat:.2f}, p = {p_val:.2e}")

In [ ]:
model.summary()

In [ ]:
fig,ax = plt.subplots(1,1, figsize=(10, 6))

order=['U08',"U09","U10",'U11','U12','U13','U14','U15','U16','U17','U22' ]
labelorder = ['<8', "8-9", "9-10", "10-11", "11-12", "12-13", "13-14", "14-15", "15-16", "16-17", "17-22"]
hbn_colors=["#FF8400", "#FFCA91" ]
sns.barplot(data=temp,x='AgeGroup',y='score', hue='task',order=order,palette=hbn_colors,edgecolor='black', linewidth=1,alpha=0.8,legend=False)
g=sns.stripplot(data=temp,x='AgeGroup',y='score', hue='task', dodge=True, order=order,legend=False,palette=hbn_colors,edgecolor='black', linewidth=0.3,size=3)
g.set(title='Average IDE by Age Group and Task', ylabel='Average IDE', xlabel='',xticklabels=labelorder, ylim=(0,22), yticks=np.arange(0,25,5))
sns.despine()
plt.savefig('main_plots/HBN_rest_movie_avg_ide.pdf', transparent=True, format='pdf')

In [ ]:
# Plot the relationship between age and rest / movie FD
par_df = pd.read_csv('HBN/basic_cohort_info.csv', index_col=0)
par_df['include_both_tasks'] = [1 if (int(np.isnan(row.movie_FD))+int(np.isnan(row.rest_FD))+int(row.cleaning_done==0) == 0) else 0 for _, row in par_df.iterrows()]
print(f'number of 0 include_both_tasks: {(par_df["include_both_tasks"]==0).sum()}; length of file: {par_df.shape[0]}')
print(f'number of participants with cleaning_done 0 {par_df[par_df["cleaning_done"]==0].shape}')
par_df = par_df[(par_df['include_both_tasks']==1)]# & (par_df["cleaning_done"]==1)]
print(f'Number of subjects with both rest and movie FD: {par_df.shape[0]}')

# Exclude subjects based on  >=2mm movement for >= 1/3 timepoints
thresh=2
par_df = par_df[(par_df['rest_'+str(thresh)+'mm_perc'] < 1/3) & (par_df['movie_'+str(thresh)+'mm_perc'] < 1/3)]
print(f'Number of subjects after excluding based on {thresh}mm threshold: {par_df.shape[0]}')

In [ ]:
par_df.head()

In [ ]:
# Create a difference df where columns are isc, movie_ide (tphate), rest_ide (tphate), difference_ide (rest - movie), agegroup, age, rest_fd, movie_fd, session
data_list = []
for subject in results_df['subject'].unique():
    subj_df = results_df[results_df['subject']==subject]
    agegroup = subj_df['AgeGroup'].values[0]
    age = subj_df['Age'].values[0]
    try:
        movie_fd = par_df[par_df['subject_id']==subject]['movie_FD'].values[0]
        rest_fd = par_df[par_df['subject_id']==subject]['rest_FD'].values[0]
        session = par_df[par_df['subject_id']==subject]['session'].values[0]
        sex = par_df[par_df['subject_id']==subject]['sex'].values[0]
    except:
        print(f'Could not find FD info for subject {subject}, skipping...')
        continue
    # get ISC for all regions
    isc_values = subj_df[subj_df['measure']=='ISC'][['region_name','score']].set_index('region_name')['score'].to_dict()
    # get movie IDE for all regions
    movie_ide_values = subj_df[(subj_df['measure']=='TPHATE_DiffOp') & (subj_df['task']=='movieTP')].set_index('region_name')['score'].to_dict()
    # get rest IDE for all regions
    rest_ide_values = subj_df[(subj_df['measure']=='TPHATE_DiffOp') & (subj_df['task']=='rest')].set_index('region_name')['score'].to_dict()
    # Combine into a dataframe
    for region in isc_values.keys():
        isc = isc_values[region]
        movie_ide = movie_ide_values[region]
        rest_ide = rest_ide_values[region]
        difference_ide = rest_ide - movie_ide
        data_list.append({'subject': subject, 'region_name': region, 'ISC': isc, 
                          'movie_score': movie_ide, 'rest_score': rest_ide, 'RestMovieDiff': difference_ide, 
                          'AgeGroup': agegroup, 'Age': age, 'movie_FD': movie_fd, 'sex':sex, 'rest_FD': rest_fd, 'session': session})
# Convert to dataframe
diff_df = pd.DataFrame(data_list)
diff_df.reset_index(inplace=True, drop=True)
diff_df.to_csv(f'{hu.get_results_dir()}/parcelwise_difference_ISC_IDE.csv')    


In [ ]:
diff_df = pd.read_csv(f'{hu.get_results_dir()}/parcelwise_difference_ISC_IDE.csv')    


In [ ]:
diff_df.head()

In [ ]:
model.model?

In [ ]:
model = smf.mixedlm('RestMovieDiff ~ ISC*Age + movie_FD + rest_FD', data=diff_df, groups=diff_df['subject']).fit(reml=False)
print(model.summary())
# Calculate the variance explained by this model and then by each fixed effect
full_r2 =
print(f'Full model R^2: {full_r2:.4f}')
# To calculate the variance explained by each fixed effect, we can fit reduced models that exclude each effect and compare the R^2 values
effects = ['ISC', 'Age', 'ISC:Age']
for effect in effects:
    reduced_formula = 'RestMovieDiff ~ ' + ' + '.join([e for e in effects if e != effect]) + ' + movie_FD + rest_FD'
    reduced_model = smf.mixedlm(reduced_formula, data=diff_df, groups=diff_df['subject']).fit(reml=False)
    reduced_r2 = reduced_model.rsquared
    effect_r2 = full_r2 - reduced_r2
    print(f'Variance explained by {effect}: {effect_r2:.4f}')

# Exploration of participant inclusion and motion stats

In [ ]:

# Look at the relationshp between age and FD, add a third column showing distributions and within-subject test
fig, axes = plt.subplots(1,3, figsize=(15,5), sharex=False, sharey=True)

# Rest FD (axes[0])
mask = par_df[['age','rest_FD']].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, 'rest_FD']
sns.regplot(data=par_df.loc[mask], x='age', y='rest_FD', ax=axes[0],
            scatter_kws={'s':20, 'alpha':0.2}, ci=None)
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
axes[0].set_title('Rest FD vs Age')
axes[0].text(0.98, 0.98, f'n={n_overall}\nr={r_overall:.2f}\np={p_overall:.3g}',
             transform=axes[0].transAxes, ha='right', va='top')

# Movie FD (axes[1])
mask = par_df[['age','movie_FD']].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, 'movie_FD']
sns.regplot(data=par_df.loc[mask], x='age', y='movie_FD', ax=axes[1],
            scatter_kws={'s':20, 'alpha':0.2}, ci=None)
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
axes[1].set_title('Movie TP FD vs Age')
axes[1].text(0.98, 0.98, f'n={n_overall}\nr={r_overall:.2f}\np={p_overall:.3g}',
             transform=axes[1].transAxes, ha='right', va='top')

# Distribution / paired comparison (axes[2])
ax3 = axes[2]
paired = par_df[['subject_id','rest_FD','movie_FD']].dropna()  # keep only subjects with both
df_long = pd.melt(paired, id_vars='subject_id', value_vars=['rest_FD','movie_FD'],
                  var_name='task', value_name='FD')
# violin + points + paired lines
sns.violinplot(data=df_long, x='task', y='FD', ax=ax3,
               palette={'rest_FD':'C0','movie_FD':'C1'}, inner=None, cut=0)
sns.boxplot(data=df_long, x='task', y='FD', ax=ax3, width=0.15, showcaps=True,
            boxprops={'zorder':2}, showfliers=False, whiskerprops={'linewidth':1})
sns.stripplot(data=df_long, x='task', y='FD', ax=ax3, color='k', size=3, jitter=True, alpha=0.6, zorder=3)

# connect paired points
if paired.shape[0] > 0:
    # map task to x positions used by violin/boxplot: 0 -> rest_FD, 1 -> movie_FD
    for _, row in paired.iterrows():
        ax3.plot([0,1], [row['rest_FD'], row['movie_FD']], color='gray', alpha=0.3, linewidth=0.5, zorder=1)

# within-subject statistics: paired t-test and Wilcoxon
if paired.shape[0] > 1:
    t_stat, p_t = stats.ttest_rel(paired['rest_FD'], paired['movie_FD'], nan_policy='omit')
    try:
        w_stat, p_w = wilcoxon(paired['rest_FD'], paired['movie_FD'])
    except Exception:
        p_w = np.nan
    ax3.text(0.5, 0.95, f'Paired t-test: t={t_stat:.2f}, p={p_t:.3g}\nWilcoxon p={p_w:.3g}',
             transform=ax3.transAxes, ha='center', va='top')
    print(f'Paired comparison (n={paired.shape[0]}): paired t-test t={t_stat:.4f}, p={p_t:.6g}; Wilcoxon p={p_w:.6g}')
else:
    ax3.text(0.5, 0.95, 'Insufficient paired data', transform=ax3.transAxes, ha='center', va='top')
    print('Insufficient paired data for within-subject test.')

ax3.set_xticklabels(['Rest', 'Movie'])
ax3.set_title('FD distribution (paired rest vs movie)')
ax3.set_ylabel('Framewise displacement (FD)')

plt.tight_layout()


In [ ]:
# Single histogram of ages by sex (all sites) using existing precomputed bins/counts/palette variables
plt.figure(figsize=(8,4))
width=0.4
plt.bar(bin_centers - width/3, counts_m, width=width, color=palette['Male'], edgecolor='black', label=f"Male (n={len(ages_m)})")
plt.bar(bin_centers,              counts_f, width=width, color=palette['Female'], edgecolor='black', label=f"Female (n={len(ages_f)})")
plt.bar(bin_centers + width/3, counts_o, width=width, color=palette['Other'], edgecolor='black', label=f"Other (n={len(ages_o)})")

plt.xlabel('Age (years)')
plt.ylabel('Count')
plt.title('Distribution of participant ages by sex (all sites)')
plt.xticks(bins)
plt.legend()
plt.tight_layout()

In [ ]:
# Histogram of ages by sex, one subplot per site
sites = sorted(par_df['session'].dropna().unique())
sexes = ['Male','Female','Other']  # preserve existing categories (Male, Female, Other...)
set2 = sns.color_palette('Set2')
palette = {sexes[i]:set2[i] for i in range(3)}
n_sites = 3
n_sexes = 3

# layout
ncols = 3
nrows = 1
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)  # flatten for easy indexing

# visual params
if 'bins' not in globals():
    bins = np.arange(np.floor(par_df['age'].min()), np.ceil(par_df['age'].max()) + 1, 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
# get the colors from the Set2 color palette

# fallback palette for any additional sex categories 
# width allocation for multiple sex categories
total_width = 0.8
width_ind = total_width / n_sexes
offsets = np.linspace(-total_width/2 + width_ind/2, total_width/2 - width_ind/2, n_sexes)

for ax, site in zip(axes, sites):
    site_df = par_df[par_df['session'] == site]
    # plot each sex as side-by-side bars
    for i, sex_cat in enumerate(sexes):
        ages_sex = site_df.loc[site_df['sex'] == sex_cat, 'age'].dropna()
        counts, _ = np.histogram(ages_sex, bins=bins)
        # choose color
        color = palette[sex_cat]
        ax.bar(bin_centers + offsets[i], counts, width=width_ind, color=color,
               edgecolor='black', alpha=0.8, label=f'{sex_cat} (n={len(ages_sex)})' if i==0 else f'{sex_cat} (n={len(ages_sex)})')
    ax.set_title(f'{site}')
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Count')
    ax.set_xticks(bins)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize='small')

# turn off any unused axes
for ax in axes[len(sites):]:
    ax.axis('off')

plt.tight_layout()

In [ ]:
fig2, axes2 = plt.subplots(1,4, figsize=(20,5), sharex=True)
axes2=axes2.ravel()
metrics = [
('movie_3mm_perc','Movie perc > 3mm'),('rest_3mm_perc','Rest perc > 3mm'), 
('movie_2mm_perc','Movie perc > 2mm'),
('rest_2mm_perc','Rest perc > 2mm'),
('movie_1mm_perc','Movie perc > 1mm'),
('rest_1mm_perc','Rest perc > 1mm'),]
# Plot the relationship between age and motion metrics for each mm threshold
# plot both tasks on the same plot, different colors
j=0
for i in range(3):
    # plot rest and movie on the same plot as different colors
    ax = axes2[i]
    metric_movie = metrics[j][0]
    metric_rest = metrics[j+1][0]
    label_movie = metrics[j][1]
    label_rest = metrics[j+1][1]
    print(label_rest, label_movie)
    # rest
    mask = par_df[['age',metric_rest]].notnull().all(axis=1)
    x = par_df.loc[mask, 'age']
    y = par_df.loc[mask, metric_rest]
    sns.regplot(data=par_df.loc[mask], x='age', y=metric_rest, ax=ax,
                scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C0')
    r_overall, p_overall = stats.pearsonr(x, y)
    n_overall = mask.sum()
    ax.text(0.98, 0.98, f'Rest r={r_overall:.2f}, p={p_overall:.2g}',
             transform=ax.transAxes, ha='right', va='top', color='C0')
    # movie
    mask = par_df[['age',metric_movie]].notnull().all(axis=1)
    x = par_df.loc[mask, 'age']
    y = par_df.loc[mask, metric_movie]
    sns.regplot(data=par_df.loc[mask], x='age', y=metric_movie, ax=ax,
                scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C1')
    r_overall, p_overall = stats.pearsonr(x, y)
    n_overall = mask.sum()
    ax.text(0.98, 0.90, f'Movie r={r_overall:.2f}, p={p_overall:.2g}',
             transform=ax.transAxes, ha='right', va='top', color='C1')
    ax.set_title(f'Motion metric: {label_rest.split()[2]} {label_rest.split()[3]}')
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Percentage of volumes')
    j+=2

ax=axes2[3]
metrics = [ ('movie_FD','Movie mean FD '), ('rest_FD','Rest mean FD')]
# Plot the relationship between age and motion metrics for each mm threshold
mask = par_df[['age',"rest_FD"]].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, "rest_FD"]
sns.regplot(data=par_df.loc[mask], x='age', y="rest_FD", ax=ax,
            scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C0')
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
ax.text(0.98, 0.98, f'Rest r={r_overall:.2f}, p={p_overall:.2g}',
            transform=ax.transAxes, ha='right', va='top', color='C0')
# movie
mask = par_df[['age',"movie_FD"]].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, "movie_FD"]
sns.regplot(data=par_df.loc[mask], x='age', y="movie_FD", ax=ax,
            scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C1')
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
ax.text(0.98, 0.90, f'Movie r={r_overall:.2f}, p={p_overall:.2g}',
            transform=ax.transAxes, ha='right', va='top', color='C1')
ax.set_title(f'Mean FD across timeseries')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Mean FD')



In [ ]:
fig2, ax = plt.subplots(1,1, figsize=(6,5), sharex=True)

metrics = [
('movie_3mm_perc','Movie perc > 3mm'),
('rest_3mm_perc','Rest perc > 3mm'), 
('movie_2mm_perc','Movie perc > 2mm'),
('rest_2mm_perc','Rest perc > 2mm'),
('movie_1mm_perc','Movie perc > 1mm'),
('rest_1mm_perc','Rest perc > 1mm'),]
# Plot the relationship between age and motion metrics for each mm threshold
# plot both tasks on the same plot, different colors
thresh=2
par_df = par_df[(par_df['rest_'+str(thresh)+'mm_perc'] < .3) & (par_df['movie_'+str(thresh)+'mm_perc'] < .3)]

metrics = [ ('movie_FD','Movie mean FD '), ('rest_FD','Rest mean FD')]
# Plot the relationship between age and motion metrics for each mm threshold
mask = par_df[['age',"rest_FD"]].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, "rest_FD"]
sns.regplot(data=par_df.loc[mask], x='age', y="rest_FD", ax=ax,
            scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C0')
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
ax.text(0.98, 0.98, f'Rest r={r_overall:.2f}, p={p_overall:.2g}',
            transform=ax.transAxes, ha='right', va='top', color='C0')
# movie
mask = par_df[['age',"movie_FD"]].notnull().all(axis=1)
x = par_df.loc[mask, 'age']
y = par_df.loc[mask, "movie_FD"]
sns.regplot(data=par_df.loc[mask], x='age', y="movie_FD", ax=ax,
            scatter_kws={'s':20, 'alpha':0.2}, ci=None, label=None, color='C1')
r_overall, p_overall = stats.pearsonr(x, y)
n_overall = mask.sum()
ax.text(0.98, 0.90, f'Movie r={r_overall:.2f}, p={p_overall:.2g}',
            transform=ax.transAxes, ha='right', va='top', color='C1')
ax.set_title(f'Mean FD across timeseries')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Mean FD')



In [ ]:

# Second row: separate regressions for Male and Female on the same plot
fig2, axes2 = plt.subplots(1,2, figsize=(10,5), sharex=True, sharey=True)
metrics = [('rest_FD','Rest FD'), ('movie_FD','Movie TP FD')]
colors = {'Male':'C0','Female':'C1'}

for i, (metric, title) in enumerate(metrics):
    ax = axes2[i]
    # overall (light grey) for context
    mask_all = par_df[['age', metric]].notnull().all(axis=1)
    # if mask_all.sum() > 1:
    #     sns.regplot(data=par_df.loc[mask_all], x='age', y=metric, ax=ax,
    #                 scatter=False, line_kws={'color':'lightgrey','linewidth':1}, ci=None)
    ax.set_title(f'{title} vs Age')
    y_text_pos = 0.98

    # plot Male and Female
    for sex,color in palette.items():
        mask = par_df[['age', metric, 'sex']].notnull().all(axis=1) & (par_df['sex'] == sex)
        n = mask.sum()
        if n > 1:
            sns.regplot(data=par_df.loc[mask], x='age', y=metric, ax=ax,
                        scatter_kws={'s':20, 'alpha':0.6,'color':color}, 
                        line_kws={'color':color}, ci=None, label=f'{sex} (n={n})')
            r, p = stats.pearsonr(par_df.loc[mask,'age'], par_df.loc[mask,metric])
            # annotate per-sex correlation on the plot
            ax.text(0.98, y_text_pos, f'{sex[0]}: r={r:.2f}, p={p:.4f}',
                    transform=ax.transAxes, ha='right', va='top', color=color)
            y_text_pos -= 0.12
            # also print to stdout
        else:
            print(f'{title} - {sex}: insufficient data (n={n})')

    ax.legend()
plt.tight_layout()


In [ ]:
def get_age_groupings(age):
    if age < 10:
        return 'U10'
    if age < 15:
        return '10-14'
    if age < 20:
        return '15-19'
    return 'O20'

results_df['AgeGroup1'] = [get_age_groupings(a) for a in results_df['Age']]


# BACKUP (ORIGINAL) Run analyses looking at the change in ID, ISC, and ID ~ ISC over age

In [ ]:
# Subject-level: barplot by AgeGroup (left) and partial scatter/regression controlling for movie_FD (right)

# recompute subject-wise mean ISC (across parcels) and merge demo/motion info
task = 'movieTP'
measure = 'ISC'
df_regional = results_df[(results_df['task'] == task) & (results_df['measure'] == measure)].copy()
sub_mean = df_regional.groupby('subject')['score'].mean().reset_index().rename(columns={'score': 'ISC_mean'})

# merge age and movie_FD from par_df
meta = par_df[['subject_id','movie_FD','AgeGroup1','age']].rename(columns={'subject_id': 'subject', 'age':'Age', 'AgeGroup1':'AgeGroup'})
# create consistent age groups using the helper already defined in the notebook
#meta['AgeGroup'] = meta['Age'].apply(get_age_groupings)
sub_df = sub_mean.merge(meta, on='subject', how='left').dropna(subset=['ISC_mean', 'movie_FD', 'AgeGroup'])
print(f"Subjects in analysis: {len(sub_df)}")

# compute group stats for bar plot
age_order = sorted(sub_df.AgeGroup.unique())#['U10','10-14','15-19','O20']
grp = sub_df.groupby('AgeGroup').agg(mean_ISC=('ISC_mean','mean'),
                                     sem_ISC=('ISC_mean','sem'),
                                     n=('ISC_mean','count')).reindex(age_order).dropna(how='all')
# plotting: left = barplot of mean ISC per AgeGroup, right = partial scatter/regression (control movie_FD)
fig, axes = plt.subplots(1,2, figsize=(12,5), gridspec_kw={'width_ratios':[1,1.1]},sharey=True)

# Left: barplot with errorbars and sample sizes
ax = axes[0]
x = np.arange(len(grp))
bar_cols = helper.get_palette12_rainbow() #sns.color_palette('Set2', n_colors=len(grp))
ax.bar(x, grp['mean_ISC'], yerr=grp['sem_ISC'], color=bar_cols, edgecolor='k')
ax.set_xticks(x)
ax.set_xticklabels(grp.index)
ax.set_xlabel('Age group')
ax.set_ylabel('Mean ISC')
ax.set_title('Mean ISC by Age group')
# annotate counts
for xi, (m, n) in enumerate(zip(grp['mean_ISC'], grp['n'])):
    ax.text(xi, 0, f"{int(n)}",
            ha='center', va='bottom', fontsize=9)

# Right: partial relationship ISC_mean ~ Age controlling for movie_FD
# Fit full model and also residualize ISC_mean by movie_FD (partial y)
model = smf.ols('ISC_mean ~ Age + movie_FD', data=sub_df).fit()
# residualize ISC_mean by movie_FD
res_y = smf.ols('ISC_mean ~ movie_FD', data=sub_df).fit().resid

ax2 = axes[1]
# scatter raw points colored by AgeGroup for context
palette_local = {g: c for g,c in zip(age_order, sns.color_palette('Set2', n_colors=len(age_order)))}
for grp_name, dfg in sub_df.groupby('AgeGroup'):
    ax2.scatter(dfg['Age'], dfg['ISC_mean'], label=f'{grp_name} (n={len(dfg)})', alpha=0.7, s=30,
                color=bar_cols[age_order.index(grp_name)])

# plot regression line for the Age effect controlling movie_FD:
# predict ISC at mean movie_FD across a dense Age grid
age_grid = np.linspace(sub_df['Age'].min(), sub_df['Age'].max(), 100)
pred_df = pd.DataFrame({'Age': age_grid, 'movie_FD': sub_df['movie_FD'].mean()})
pred = model.predict(pred_df)
ax2.plot(age_grid, pred, color='k', linewidth=2, label='Age effect (at mean movie_FD)')

# also plot partial (residualized) regression for clarity: residuals vs Age
sns.regplot(x=sub_df['Age'], y=res_y, scatter=False, ci=95, ax=ax2, color='k', line_kws={'linestyle':'--', 'alpha':0.6})

# annotate model Age coefficient and p-value
coef_age = model.params.get('Age', np.nan)
p_age = model.pvalues.get('Age', np.nan)
ax2.text(0.05, 0.95, f'Age coef={coef_age:.4f}\np={p_age:.3g}\nN={len(sub_df)}',
         transform=ax2.transAxes, ha='left', va='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax2.set_xlabel('Age (years)')
ax2.set_ylabel('')
ax2.set_title('Mean ISC ~ Age (controlling for movie_FD)')
# ax2.legend(fontsize='small', loc='lower left', ncol=1)
sns.despine()
plt.tight_layout()


In [ ]:
# Subject-level analyses for TPHATE_DiffOp (IDE) and PCA (mean across parcels) analogous to the ISC analysis above
task = 'movieTP'
measures_to_run = ['TPHATE_DiffOp', 'PCA']

for task,var in zip(['movieTP','rest'],['movie_FD','rest_FD']):
    for measure in measures_to_run:
        df_regional = results_df[(results_df['task'] == task) & (results_df['measure'] == measure)].copy()
        sub_mean = df_regional.groupby('subject')['score'].mean().reset_index().rename(columns={'score': f'{measure}_mean'})

        # merge age and movie_FD from par_df (reuse same meta columns as earlier)
        meta = par_df[['subject_id',var,'AgeGroup1','age','sex','session']].rename(columns={'subject_id': 'subject', 'age':'Age', 'AgeGroup1':'AgeGroup'})
        sub_df = sub_mean.merge(meta, on='subject', how='left').dropna(subset=[f'{measure}_mean', var, 'AgeGroup','sex','session'])

        # group stats for bar plot
        age_order = sorted(sub_df.AgeGroup.unique())
        grp = sub_df.groupby('AgeGroup').agg(mean_val=(f'{measure}_mean','mean'),
                                            sem_val=(f'{measure}_mean','sem'),
                                            n=('subject','count')).reindex(age_order).dropna(how='all')

        # plotting: left = barplot by AgeGroup, right = partial scatter/regression controlling for movie_FD
        fig, axes = plt.subplots(1,2, figsize=(12,5), gridspec_kw={'width_ratios':[1,1.1]}, sharey=True)
        ax = axes[0]
        x = np.arange(len(grp))
        bar_cols = helper.get_palette12_rainbow()#sns.color_palette('Set2', n_colors=len(grp))
        ax.bar(x, grp['mean_val'], yerr=grp['sem_val'], color=bar_cols, edgecolor='k')
        ax.set_xticks(x)
        ax.set_xticklabels(grp.index)
        ax.set_xlabel('Age group')
        ax.set_ylabel(f'Mean {measure}')
        ax.set_title(f'Mean {measure} by Age group: {task}')
        for xi, (m, n) in enumerate(zip(grp['mean_val'], grp['n'])):
            ax.text(xi, 0, f"{int(n)}", ha='center', va='bottom', fontsize=9)

        # Right: partial relationship measure_mean ~ Age controlling for movie_FD
        model = smf.ols(f'{measure}_mean ~ Age + {var} ', data=sub_df).fit()
        # residualize y by movie_FD for a partial-plot feel
        res_y = smf.ols(f'{measure}_mean ~ {var} ', data=sub_df).fit().resid

        ax2 = axes[1]
        palette_local = {g: c for g,c in zip(age_order, sns.color_palette('Set2', n_colors=len(age_order)))}
        for grp_name, dfg in sub_df.groupby('AgeGroup'):
            ax2.scatter(dfg['Age'], dfg[f'{measure}_mean'], label=f'{grp_name} (n={len(dfg)})', alpha=0.7, s=30,
                        color=bar_cols[age_order.index(grp_name)])

        # # predict at mean movie_FD across Age grid
        age_grid = np.linspace(sub_df['Age'].min(), sub_df['Age'].max(), 100)
        pred_df = pd.DataFrame({'Age': age_grid, var: sub_df[var].mean()})
        pred = model.predict(pred_df)
        ax2.plot(age_grid, pred, color='k', linewidth=2, label=f'Age effect (at mean {var})')

        coef_age = model.params.get('Age', np.nan)
        p_age = model.pvalues.get('Age', np.nan)
        ax2.text(0.05, 0.95, f'Age coef={coef_age:.4f}\np={p_age:.3g}\nN={len(sub_df)}',
                transform=ax2.transAxes, ha='left', va='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

        ax2.set_xlabel('Age (years)')
        ax2.set_ylabel('')
        ax2.set_title(f'Mean {measure} ~ Age (controlling for {var}): {task}')
        sns.despine()
        plt.tight_layout()
        plt.show()

In [ ]:
# Subject-level analyses for TPHATE_DiffOp (IDE) and PCA (standard deviation across parcels) analogous to the ISC analysis above
measures_to_run = ['TPHATE_DiffOp', 'PCA']

for task,var in zip(['movieTP','rest'],['movie_FD','rest_FD']):
    for measure in measures_to_run:
        df_regional = results_df[(results_df['task'] == task) & (results_df['measure'] == measure)].copy()
        sub_std = df_regional.groupby('subject')['score'].std().reset_index().rename(columns={'score': f'{measure}_std'})

        # merge age and movie_FD from par_df (reuse same meta columns as earlier)
        meta = par_df[['subject_id',var,'AgeGroup1','age']].rename(columns={'subject_id': 'subject', 'age':'Age', 'AgeGroup1':'AgeGroup'})
        sub_df = sub_std.merge(meta, on='subject', how='left').dropna(subset=[f'{measure}_std', var, 'AgeGroup'])

        # group stats for bar plot
        age_order = sorted(sub_df.AgeGroup.unique())
        grp = sub_df.groupby('AgeGroup').agg(mean_val=(f'{measure}_std','mean'),
                                            sem_val=(f'{measure}_std','sem'),
                                            n=('subject','count')).reindex(age_order).dropna(how='all')

        # plotting: left = barplot by AgeGroup, right = partial scatter/regression controlling for movie_FD
        fig, axes = plt.subplots(1,2, figsize=(12,5), gridspec_kw={'width_ratios':[1,1.1]}, sharey=True)
        ax = axes[0]
        x = np.arange(len(grp))
        bar_cols = helper.get_palette12_rainbow()#sns.color_palette('Set2', n_colors=len(grp))
        ax.bar(x, grp['mean_val'], yerr=grp['sem_val'], color=bar_cols, edgecolor='k')
        ax.set_xticks(x)
        ax.set_xticklabels(grp.index)
        ax.set_xlabel('Age group')
        ax.set_ylabel(f'Standard deviation {measure}')
        ax.set_title(f'Standard deviation {measure} by Age group: {task}')
        for xi, (m, n) in enumerate(zip(grp['mean_val'], grp['n'])):
            ax.text(xi, 0, f"{int(n)}", ha='center', va='bottom', fontsize=9)

        # Right: partial relationship measure_mean ~ Age controlling for movie_FD
        model = smf.ols(f'{measure}_std ~ Age + {var}', data=sub_df).fit()
        # residualize y by movie_FD for a partial-plot feel
        res_y = smf.ols(f'{measure}_std ~ {var}', data=sub_df).fit().resid

        ax2 = axes[1]
        palette_local = {g: c for g,c in zip(age_order, sns.color_palette('Set2', n_colors=len(age_order)))}
        for grp_name, dfg in sub_df.groupby('AgeGroup'):
            ax2.scatter(dfg['Age'], dfg[f'{measure}_std'], label=f'{grp_name} (n={len(dfg)})', alpha=0.7, s=30,
                        color=bar_cols[age_order.index(grp_name)])

        age_grid = np.linspace(sub_df['Age'].min(), sub_df['Age'].max(), 100)
        pred_df = pd.DataFrame({'Age': age_grid, var: sub_df[var].std()})
        pred = model.predict(pred_df)
        ax2.plot(age_grid, pred, color='k', linewidth=2, label=f'Age effect (at mean {var})')

        coef_age = model.params.get('Age', np.nan)
        p_age = model.pvalues.get('Age', np.nan)
        ax2.text(0.05, 0.95, f'Age coef={coef_age:.4f}\np={p_age:.3g}\nN={len(sub_df)}',
                transform=ax2.transAxes, ha='left', va='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

        ax2.set_xlabel('Age (years)')
        ax2.set_ylabel('')
        ax2.set_title(f'Std {measure} ~ Age (controlling for {var}): {task}')
        sns.despine()
        plt.tight_layout()
        plt.show()

In [ ]:
# Aggregate ISC and TPHATE_DiffOp (IDE) across participants by AgeGroup and plot with the helper.
task = 'movieTP'
measures = {'ISC': 'ISC', 'TPHATE_DiffOp': 'IDE'}
results_dfa = results_df[results_df['task'] == task]
df = results_dfa[results_dfa['measure'].isin(measures.keys())].copy()

# mean score per region x AgeGroup x measure
# make sure that the mean is excluding nans
mean_by = df.groupby(['AgeGroup', 'measure', 'region_name'])['score'].agg(lambda x: x.mean(skipna=True)).reset_index()

age_groups = sorted(mean_by['AgeGroup'].unique())
data_arrs = []
titles = []
img_fns = []

# create a data array (Series indexed by region_name) for each AgeGroup x measure
for m in measures.keys():  # order: ISC then TPHATE_DiffOp
    for age in age_groups:
        s = mean_by[(mean_by['AgeGroup'] == age) & (mean_by['measure'] == m)].set_index('region_name')['score']
        # ensure a consistent index type (Series is acceptable for helper functions that map by region name)
        data_arrs.append(s)
        titles.append(f"{age} {measures[m]}")
        img_fns.append(f'HBN/plots/{task}_{age}_{m}.png')  # placeholder, let helper handle atlas internally

n_rows = len(age_groups)  # each row will contain ISC and IDE for one AgeGroup
# output path for the compiled figure
output_path = os.path.join(hu.get_results_dir(), 'all_participants_agegroup_ISC_TPHATE_surface_grid.png')

# helper.compile_surface_plots_to_grid_by_rows(img_fns,data_arrs, output_path, n_rows=2, atlas='searchlight',rerun=False,titles=['aeronaut ISC', 'mickey ISC', 'aeronaut IDE', 'mickey IDE'],
#                                              cbar_ranges_per_row=[[0.0, 0.5], [3, 18]], cbar_labels_per_row=['ISC', 'IDE'], cmaps_per_row=['magma', 'viridis'])
# call the plotting helper
helper.compile_surface_plots_to_grid_by_rows(img_fns, data_arrs, output_path, n_rows=2, surf_type='fsaverage', target_density='41k', method='nearest',  atlas='Schaefer', rerun=True, cmaps_per_row=['magma','viridis'],
                                             cbar_ranges_per_row=[[0.0, 0.4], [1, 18]], cbar_labels_per_row=['ISC', 'IDE'],
                                             titles=titles)

In [ ]:
# Aggregate ISC and TPHATE_DiffOp (IDE) across participants by AgeGroup and plot with the helper.
def get_age_groupings(age):
    if age < 10:
        return 'U10'
    if age < 15:
        return '10-14'
    if age < 20:
        return '15-19'
    return 'O20'

task = 'movieTP'
measures = {'ISC': 'ISC', 'TPHATE_DiffOp': 'IDE'}
results_dfa = results_df[(results_df['task'] == task) ]
df = results_dfa[results_dfa['measure'].isin(measures.keys())].copy()
# do new age groupings age_groups =  #$sorted(mean_by['AgeGroup1'].unique())
df['AgeGroup1'] = [get_age_groupings(a) for a in df['Age']]
# mean score per region x AgeGroup x measure
# make sure that the mean is excluding nans
mean_by = df.groupby(['AgeGroup1', 'measure', 'region_name'])['score'].agg(lambda x: x.mean(skipna=True)).reset_index()


data_arrs = []
titles = []
img_fns = []

# create a data array (Series indexed by region_name) for each AgeGroup x measure
for m in measures.keys():  # order: ISC then TPHATE_DiffOp
    for age in ['U10','10-14','15-19','O20']:
        s = mean_by[(mean_by['AgeGroup1'] == age) & (mean_by['measure'] == m)].set_index('region_name')['score']
        # ensure a consistent index type (Series is acceptable for helper functions that map by region name)
        data_arrs.append(s)
        titles.append(f"{age} {measures[m]}")
        img_fns.append(f'HBN/plots/{task}_{age}_{m}.png')  # placeholder, let helper handle atlas internally

n_rows = len(age_groups)  # each row will contain ISC and IDE for one AgeGroup
# output path for the compiled figure
output_path = os.path.join(hu.get_results_dir(), 'motion_excl_agegroup_ISC_TPHATE_surface_grid.png')

# helper.compile_surface_plots_to_grid_by_rows(img_fns,data_arrs, output_path, n_rows=2, atlas='searchlight',rerun=False,titles=['aeronaut ISC', 'mickey ISC', 'aeronaut IDE', 'mickey IDE'],
#                                              cbar_ranges_per_row=[[0.0, 0.5], [3, 18]], cbar_labels_per_row=['ISC', 'IDE'], cmaps_per_row=['magma', 'viridis'])
# call the plotting helper
helper.compile_surface_plots_to_grid_by_rows(img_fns, data_arrs, output_path, n_rows=2, surf_type='fsaverage', target_density='41k', method='nearest',    
                                             atlas='Schaefer', rerun=True, cmaps_per_row=['magma','viridis'],
                                             cbar_ranges_per_row=[[0.0, 0.3], [2, 17]], cbar_labels_per_row=['ISC', 'IDE'],
                                             titles=titles)

In [ ]:
def parcel_age_regressions(df, region_order=None, covariates=[None], participant_df=None):  
    region_order = region_order if region_order is not None else sorted(df['region_name'].unique())
    if covariates[0] is not None and covariates[0] not in df.columns:
        # look in the participant df 
        if participant_df is not None:
            temp=participant_df[[ 'subject_id'] + covariates]
            df = df.merge(temp, left_on='subject', right_on='subject_id', how='left')
        else:
            print(f"Covariate {covariates[0]} not found in df columns and no participant_df provided.")
            return
    rows = []
    for reg in region_order:
        df_reg = df[df['region_name'] == reg]
        # Check for Covariates
        if covariates[0] is None:
            formula = 'score ~ Age'
        else:
            formula = f'score ~ Age '+ ''.join([f' + {cov}' for cov in covariates])
        
        model = smf.ols(formula, data=df_reg).fit()
        # Now get the reults of this model
        coef = model.params['Age']
        tval = model.tvalues['Age']
        pval = model.pvalues['Age']
        r2 = model.rsquared
        rows.append({ 'region_name': reg, 'coef': coef, 't': tval,  'p':pval, 'r2': r2})
    res_df = pd.DataFrame(rows)
    if res_df.shape[0] > 0:
        reject, pvals_fdr, _, _ = multipletests(res_df['p'].values, alpha=0.05, method='fdr_bh')
        res_df['p_fdr'] = pvals_fdr
        res_df['sig_fdr'] = reject
    return res_df

def parcel_age_mixed_model(df, fixed_effects, random_effects, region_order=None, participant_df=None):
    region_order = region_order if region_order is not None else sorted(df['region_name'].unique())
    covariates = [fe for fe in fixed_effects] + [re for re in random_effects]
    if covariates[0] is not None and covariates[0] not in df.columns:
        # look in the participant df 
        if participant_df is not None:
            temp=participant_df[[ 'subject_id'] + covariates]
            df = df.merge(temp, left_on='subject', right_on='subject_id', how='left')
        else:
            print(f"Covariate {covariates[0]} not found in df columns and no participant_df provided.")
            return
    print(df.columns.tolist())
    # turn random effect to int
    unique_vals = df[random_effects[0]].unique()
    val_to_int = {val: i for i, val in enumerate(unique_vals)}
    
    print(df.shape)
    rows = []
    # Build formula
    fixed_formula = ' + '.join(fixed_effects)
    formula = f'score ~ {fixed_formula} '
    print(f"Mixed model formula: {formula}")
    for reg in region_order:
        df_reg = df[df['region_name'] == reg].reset_index(drop=True)
        print(df_reg.shape, np.unique(df_reg[random_effects[0]].values, return_counts=True))
        model = smf.mixedlm(formula, data=df_reg, groups=df_reg[random_effects[0]]).fit()
        # Now get the results of this model
        coef = model.params['Age']
        tval = model.tvalues['Age']
        pval = model.pvalues['Age']
        r2 = model.rsquared if hasattr(model, 'rsquared') else np.nan
        rows.append({ 'region_name': reg,  'coef': coef, 't': tval,  'p':pval, 'r2': r2})
    res_df = pd.DataFrame(rows)
    if res_df.shape[0] > 0:
        reject, pvals_fdr, _, _ = multipletests(res_df['p'].values, alpha=0.05, method='fdr_bh')
        res_df['p_fdr'] = pvals_fdr
        res_df['sig_fdr'] = reject
    return res_df


def determine_colorbar_range(vals):
    # get global min/max for color scaling
    vmin = np.nanmin(vals)
    vmax = np.nanmax(vals)
    # round these to even numbers for better colorbar ticks
    vmin_rounded = np.floor(vmin*10)/10
    vmax_rounded = np.ceil(vmax*10)/10
    print(f"Rounded colorbar range: {vmin_rounded} to {vmax_rounded}")
    # if zero is between these, adjust to be symmetric
    if vmin_rounded < 0 < vmax_rounded:
        abs_max = max(abs(vmin_rounded), abs(vmax_rounded))
        vmin_rounded = -abs_max
        vmax_rounded = abs_max
        print(f"Adjusted to symmetric colorbar range: {vmin_rounded} to {vmax_rounded}")
        cmap = helper.diverging_colormap_bp()
    else:
        cmap = 'viridis'
    return (vmin_rounded, vmax_rounded), cmap

In [ ]:
# # Per-parcel linear regressions: IDE ~ Age (movieTP). Saves results for  motion-excluded subs.

# # min_n = 10  # minimum subjects per parcel to run regression
REGION_ORDER=results_df['region_name'].values[:400]
# # # subset ID for the current task (uses `task` variable from notebook)
for task, cov in zip(['movieTP', 'rest'], ['movie_FD', 'rest_FD']):
    df_all = results_df[(results_df['measure'] == 'TPHATE_DiffOp') & (results_df['task'] == task)].copy()
    df_all = df_all[df_all['Age']<18]
    age_nomotion = parcel_age_regressions(df_all, region_order=REGION_ORDER, participant_df=par_df, covariates=[cov, 'sex','session'])

    print(f"Total parcels tested (motion-excluded): {len(age_nomotion)}; significant after FDR: {age_nomotion['sig_fdr'].sum() if not age_nomotion.empty else 0}")

    # Visualize ID ~ Age slope results for all subjects
    for name in ['t','coef','r2']:    # select the region order used earlier (sorted unique region names from df_isc_all)
        vals = age_nomotion.set_index('region_name').reindex(REGION_ORDER)[name].values
        vals_mask = vals * age_nomotion.set_index('region_name').reindex(REGION_ORDER)['sig_fdr'].values
        cbar_range, cmap = determine_colorbar_range(vals)
        fn = os.path.join(hu.get_results_dir(), f'age_predicts_IDE_{task}_{name}_surface.png')
        helper.generate_surface_plot(vals_mask, image_fn=fn, atlas='Schaefer', cmap=cmap, cbar_range=cbar_range, surf_type='fsaverage', target_density='41k',  
                                                                 include_cbar=True, title=f'IDE {task} ~ Age {name} masked',method='nearest', threshold=None, mask_medial_wall=True)

In [ ]:
# # Per-parcel linear regressions: ISC ~ Age (movieTP). Saves results for  motion-excluded subs.

# # min_n = 10  # minimum subjects per parcel to run regression
REGION_ORDER=results_df['region_name'].values[:400]
# # # subset ISC for the current task (uses `task` variable from notebook)
task = 'movieTP'
cov = 'movie_FD'
df_all = results_df[(results_df['measure'] == 'ISC') & (results_df['task'] == task)].copy()
age_nomotion = parcel_age_regressions(df_all, region_order=REGION_ORDER, participant_df=par_df, covariates=[cov, 'sex','session'])

print(f"Total parcels tested (motion-excluded): {len(age_nomotion)}; significant after FDR: {age_nomotion['sig_fdr'].sum() if not age_nomotion.empty else 0}")

# Visualize ISC ~ Age slope results for all subjects
for name in ['t','coef','r2']:    # select the region order used earlier (sorted unique region names from df_isc_all)
    vals = age_nomotion.set_index('region_name').reindex(REGION_ORDER)[name].values
    vals_mask = vals * age_nomotion.set_index('region_name').reindex(REGION_ORDER)['sig_fdr'].values
    cbar_range, cmap = determine_colorbar_range(vals)

    fn = os.path.join(hu.get_results_dir(), f'age_predicts_ISC_{task}_{name}_surface.png')
    helper.generate_surface_plot(vals_mask, image_fn=fn,atlas='Schaefer', cmap=cmap, cbar_range=cbar_range, surf_type='fsaverage', target_density='41k',  
                                                                include_cbar=True, title=f'ISC {task} ~ Age {name} masked',method='nearest', threshold=None, mask_medial_wall=True)

In [ ]:
def parcel_ide_isc_regressions(df, y='TPHATE_DiffOp', x='ISC', region_order=None, covariates=['movie_FD', 'Age', 'sex', 'session'], interactions=[None], participant_df=None):
    # pivot the df to have both ISC and IDE scores per subject x region
    df_pivot = df.pivot_table(index=['subject','region_name','Age'], columns='measure', values='score').reset_index()
    region_order = region_order if region_order is not None else sorted(df_pivot['region_name'].unique())
    if covariates[0] is not None and covariates[0] not in df_pivot.columns:
        # look in the participant df 
        if participant_df is not None:
            cols = [ 'subject_id'] + covariates
            if interactions[0] is not None:
                cols += [inter for inter in interactions if inter is not None]
            temp=participant_df[cols]
            df_pivot = df_pivot.merge(temp, left_on='subject', right_on='subject_id', how='left')
        else:
            print(f"Covariate {covariates[0]} not found in df columns and no participant_df provided.")
            return
    rows = []
    for reg in region_order:
        df_reg = df_pivot[df_pivot['region_name'] == reg]
        # Check for Covariates
        if covariates[0] is None and interactions[0] is None:
            formula = f'{y} ~ {x}'
        elif interactions[0] is None:
            formula = f'{y} ~ {x} '+ ''.join([f' + {cov}' for cov in covariates])
        else:
            formula = f'{y} ~ ' + ''.join([f'{x}:{inter}' for inter in interactions if inter is not None]) + ''.join([f' + {cov}' for cov in covariates])
        
        model = smf.ols(formula, data=df_reg).fit()
        # Now get the reults of this model
        coef = model.params[x]
        tval = model.tvalues[x]
        pval = model.pvalues[x]
        r2 = model.rsquared
        if interactions[0] is not None:
            for inter in interactions:
                if inter is not None:
                    inter_term = f'{x}:{inter}'
                    coef_inter = model.params[inter_term]
                    tval_inter = model.tvalues[inter_term]
                    pval_inter = model.pvalues[inter_term]
                    r2_inter = model.rsquared
                    temp = { 'region_name': reg, 'coef_interaction': coef_inter, 't_interaction': tval_inter,  'p_interaction':pval_inter, 'r2': r2_inter, 'interaction_with': inter}
        else:
            temp = {}

        temp1={ 'region_name': reg,  'coef': coef, 't': tval,  'p':pval, 'r2': r2}
        if temp != {}:
            temp1.update(temp)
        rows.append(temp1)
    res_df = pd.DataFrame(rows)
    if res_df.shape[0] > 0:
        p_col = 'p' if 'p' in res_df.columns else 'p_interaction'
        reject, pvals_fdr, _, _ = multipletests(res_df[p_col].values, alpha=0.05, method='fdr_bh')
        res_df['p_fdr'] = pvals_fdr
        res_df['sig_fdr'] = reject
    return res_df

In [ ]:
# create per-parcel within-subject difference: rest - movie for TPHATE_DiffOp
df_tphate = results_df[results_df['measure'] == 'TPHATE_DiffOp'].copy()

# be robust to task naming (e.g. 'movieTP' vs 'movie')
tasks = df_tphate['task'].dropna().unique().tolist()
movie_task = next((t for t in tasks if 'movie' in t.lower()), None)
rest_task = next((t for t in tasks if 'rest' in t.lower()), None)
if movie_task is None or rest_task is None:
    raise RuntimeError(f"Could not find both movie and rest tasks in TPHATE rows. found tasks: {tasks}")

# pivot so each row = subject x region, columns for rest and movie scores
pivot = df_tphate.pivot_table(index=['subject', 'region_name'], columns='task', values='score').reset_index()

# ensure expected columns exist
pivot = pivot.rename(columns={movie_task: 'movie_score', rest_task: 'rest_score'})

# keep only rows with both scores and compute difference (rest - movie)
pivot = pivot.dropna(subset=['movie_score', 'rest_score']).copy()
pivot['tphate_diff_rest_minus_movie'] = pivot['rest_score'] - pivot['movie_score']

# merge participant-level metrics (movie_FD, rest_FD, age)
participant_cols = ['subject_id', 'movie_FD', 'rest_FD', 'age']
meta = par_df[participant_cols].copy()
tphate_diff_df = pivot.merge(meta, left_on='subject', right_on='subject_id', how='left')

# select/arrange columns of interest
tphate_diff_df = tphate_diff_df[['subject', 'region_name', 'rest_score', 'movie_score',
                                 'tphate_diff_rest_minus_movie', 'movie_FD', 'rest_FD', 'age']]

# quick sanity output
print(f"Computed differences for {tphate_diff_df['subject'].nunique()} subjects x {tphate_diff_df['region_name'].nunique()} parcels")
tphate_diff_df.head()

In [ ]:
model = smf.ols('tphate_diff_rest_minus_movie ~ age + rest_FD + movie_FD ', data=a).fit()
model.summary()

In [ ]:
model = smf.ols('tphate_diff_rest_minus_movie ~ age + rest_FD + movie_FD + age:rest_FD + age:movie_FD', data=a).fit()
model.summary()

In [ ]:
a = tphate_diff_df.groupby('subject').mean(numeric_only=True).reset_index()
sns.regplot(x='age', y='tphate_diff_rest_minus_movie', data=a)


In [ ]:
subset = tphate_diff_df[tphate_diff_df['region_name']=='17Networks_LH_ContA_Cingm_1']
model1 = smf.ols('movie_score ~ age + movie_FD + age:movie_FD', data=subset).fit()
print(model1.summary())

model1 = smf.ols('movie_score ~ age + movie_FD', data=subset).fit()
print(model1.summary())

In [ ]:
pg.partial_corr(data=subset, x='movie_score', y='age', covar=['movie_FD'], method='spearman')


In [ ]:
# Look at more than 50% of timepoints during rest/movie where displacement is > 3mm and exclude the subject
# 3mm based on infants , then 2mm and 1mm 
# 

In [ ]:
# Load in results file
results_df = hu.load_ide_isc_atlas_df()
results_df.head()

In [ ]:
import parcelwise_regressions as pr

In [ ]:
participant_df = hu.load_participant_df()
dataframe = pr.join_results_participant_info(results_df, participant_df, 'hbn')
dataframe = dataframe[dataframe['measure'].isin(['TPHATE_DiffOp'])]
df = pr.clean_difference_dataframe(dataframe)

In [ ]:
arr.head()

In [ ]:
arr= df.groupby(['subject']).mean(numeric_only=True)
sns.regplot(x='Age', y='RestMovieDiff', data=arr, scatter_kws={'s': 10, 'alpha':0.3}, line_kws={'color': 'red'})
# add regression stats to the plot
model = smf.ols('RestMovieDiff ~ Age + movie_FD + rest_FD', data=arr).fit()
coef_age = model.params.get('Age', np.nan)
p_age = model.pvalues.get('Age', np.nan)
coef_movie_fd = model.params.get('movie_FD', np.nan)
p_movie_fd = model.pvalues.get('movie_FD', np.nan)
coef_rest_fd = model.params.get('rest_FD', np.nan)
p_rest_fd = model.pvalues.get('rest_FD', np.nan)
ax = plt.gca()
ax.text(0.05, 0.95, f'Age coef={coef_age:.3f}, p={p_age:.3f}\n'
                                         f'movie_FD coef={coef_movie_fd:.3f}, p={p_movie_fd:.3f}\n'
                                         f'rest_FD coef={coef_rest_fd:.3f}, p={p_rest_fd:.3f}\n'
                                         f'N={len(arr)}',
                transform=ax.transAxes, ha='left', va='top', 
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))


In [ ]:
arr= df.groupby(['subject']).mean(numeric_only=True)
sns.regplot(x='Age', y='movie_score', data=arr, scatter_kws={'s': 10, 'alpha':0.3}, line_kws={'color': 'red'})
# add regression stats to the plot
model = smf.ols('movie_score ~ Age + movie_FD + Age:movie_FD', data=arr).fit()
coef_age = model.params.get('Age', np.nan)
p_age = model.pvalues.get('Age', np.nan)
coef_movie_fd = model.params.get('movie_FD', np.nan)
p_movie_fd = model.pvalues.get('movie_FD', np.nan)
ax = plt.gca()
ax.text(0.05, 0.95, f'Age coef={coef_age:.3f}, p={p_age:.3f}\n'
                                         f'movie_FD coef={coef_movie_fd:.3f}, p={p_movie_fd:.3f}\n'
                                         f'N={len(arr)}',
                transform=ax.transAxes, ha='left', va='top', 
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))


In [ ]:
arr= df.groupby(['subject']).mean(numeric_only=True)
sns.regplot(x='Age', y='rest_score', data=arr, scatter_kws={'s': 10, 'alpha':0.3}, line_kws={'color': 'red'})
# add regression stats to the plot
model = smf.ols('rest_score ~ Age + rest_FD + Age:rest_FD', data=arr).fit()
coef_age = model.params.get('Age', np.nan)
p_age = model.pvalues.get('Age', np.nan)
ax = plt.gca()
ax.text(0.05, 0.95, f'Age coef={coef_age:.4f}\np={p_age:.3g}\nN={len(arr)}',
        transform=ax.transAxes, ha='left', va='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))


In [ ]:
model.summary()

# testing covariates

In [ ]:
# Plot on a grid scatterplots for Age vs movie_FD, Age vs rest_FD, movie_FD vs ISC, age vs ISC
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
# axes=axes.flatten()

for i, (x, y) in enumerate([('Age', 'movie_FD'), ('Age', 'rest_FD'), ('movie_FD', 'ISC'), ('Age', 'ISC')]):
    mask = df1[[x, y]].notnull().all(axis=1)
    sns.regplot(x=x, y=y, data=df1.loc[mask], scatter_kws={'s': 10, 'alpha':0.3}, ax=axes[i])
    r, p = stats.pearsonr(df1.loc[mask, x], df1.loc[mask, y])
    axes[i].text(0.05, 0.95, f'n={mask.sum()}\nr={r:.3f}\np={p:.3g}',transform=axes[i].transAxes, ha='left', va='top',
               bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    axes[i].set_title(f'{x} vs {y}')
sns.despine()
plt.tight_layout()
plt.show()

# IDE and ISC maps by AgeGroup

In [ ]:
# Create another age grouping
def get_age_groupings(age):
    if age < 10:
        return 'U10'
    if age < 13:
        return '10-13'
    if age < 17:
        return '14-17'
    return 'O17'
difference_df=diff_df.copy()
difference_df['AgeGroup1'] = [get_age_groupings(a) for a in difference_df['Age']]
difference_df.head()

# Now plot average ISC, average IDE movie, average IDE rest, average IDE diff by this new age grouping
difference_df_long = pd.melt(difference_df, id_vars=['subject', 'Age', 'AgeGroup1','region_name'], value_vars=['ISC', 'movie_score', 'rest_score', 'RestMovieDiff'],
                             var_name='measure', value_name='score')
difference_df_long.head(), np.unique(difference_df_long['AgeGroup1'].values, return_counts=True)




In [ ]:
REGION_ORDER = difference_df_long['region_name'].values[:400]

In [ ]:
# Plot average scores by AgeGroup1
measures = ['ISC', 'movie_score', 'rest_score', 'RestMovieDiff']
ranges = [[0.0, 0.3], [0, 18], [0, 18], [-9, 9]]
cmaps = ['magma', viridis_white, viridis_white, helper.diverging_colormap_bp()]
for i,measure in enumerate(measures):
    sub_df = difference_df_long[difference_df_long['measure'] == measure].copy()
    age_order = ['U10', '10-13', '14-17', 'O17']

    # Plot surface maps for each measure by age group
    data_arrs = []
    titles = []
    img_fns = []
    nifti_images = []
    
    for age in age_order:
        # Average across subjects in this age group
        temp = sub_df[sub_df['AgeGroup1'] == age]
        temp = temp.groupby('region_name')['score'].mean().reset_index()
        temp = temp.set_index('region_name').reindex(REGION_ORDER).reset_index()
        # Now make sure that they're in the right order for REGION_NAME
        values = temp['score'].values
        data_arrs.append(values)
        titles.append(f"{age} {measure}")
        img_fns.append(f'HBN/plots/{measure}_{age}.png')  
        
    n_rows = len(age_order)
    output_path = os.path.join(f'main_plots/HBN_all_participants_agegroup_{measure}_surface_grid.pdf')
    helper.compile_surface_plots_to_grid_by_rows(img_fns, 
                                                 data_arrs, 
                                                 output_path, 
                                                 n_rows=1, 
                                                 surf_type='fslr', 
                                                 target_density='32k',
                                                   method='linear', 
                                                   atlas='Schaefer', 
                                                   rerun=True, 
                                                   cmaps_per_row=[cmaps[i]],
                                                 cbar_ranges_per_row=[ranges[i]], 
                                                 cbar_labels_per_row=[measure],
                                                 titles=titles)


In [ ]:
# Run significance tests on the difference scores by age group
measure0,measure1 = 'rest_score','movie_score'
rangee = [-9, 9]
cmap = helper.diverging_colormap_bp()
age_order = ['U10', '10-13', '14-17', 'O17']

# Plot surface maps for each measure by age group
data_arrs = []
titles = []
img_fns = []
for age in age_order:
    temp0 = difference_df_long[(difference_df_long['AgeGroup1'] == age) & (difference_df_long['measure'] == measure0)].reset_index(drop=True)
    temp1 = difference_df_long[(difference_df_long['AgeGroup1'] == age) & (difference_df_long['measure'] == measure1)].reset_index(drop=True)
    arrays0, arrays1 = [], []
    for sub in temp0['subject'].unique():
        arr0 = temp0[temp0['subject'] == sub]['score'].values
        arr1 = temp1[temp1['subject'] == sub]['score'].values
        arrays0.append(arr0)
        arrays1.append(arr1)
    arr0 = np.array(arrays0)
    arr1 =  np.array(arrays1)
    print(f'Age group: {age}, {arr0.shape}, {arr1[0].shape}')
    # Run stats
    mean_diff, pvals, sig_mask = stats_helpers.paired_difference_ttest(arr0, arr1, alternative='two-sided', alpha=0.01)
    to_plot = mean_diff * sig_mask
    print(f'Age group: {age}, mean diff range: {mean_diff.min()} to {mean_diff.max()} , sig count: {sig_mask.sum()}')
    data_arrs.append(to_plot)
    titles.append(f"{age} {measure}")
    img_fns.append(f'HBN/plots/rmdiff_{age}_signif.png')  # placeholder, let helper handle atlas internally

n_rows = len(age_order)
output_path = os.path.join(f'main_plots/HBN_all_participants_agegroup_RMDIFF_signif_surface_grid.pdf')
helper.compile_surface_plots_to_grid_by_rows(img_fns, data_arrs, 
                                             output_path, 
                                             n_rows=1, 
                                             surf_type='fslr', 
                                                target_density='32k', 
                                                method='linear', 
                                                atlas='Schaefer', 
                                                rerun=True, 
                                                cmaps_per_row=[cmaps[i]],
                                                cbar_ranges_per_row=[ranges[i]], cbar_labels_per_row=[measure],
                                                titles=titles)


# Average across brain plots

In [ ]:
df1 = difference_df.groupby(['subject','AgeGroup']).mean(numeric_only=True).reset_index()
fig,ax=plt.subplots(2,2,figsize=(16,8),sharex=True)
ax=ax.ravel()
g=sns.barplot(x='AgeGroup', y='rest_score', data=df1, palette='magma', alpha=0.7,order=sorted(df1['AgeGroup'].unique()),ax=ax[0])
g.set(title='Rest IDE by Age Group', ylim=(0,12), ylabel='TPHATE_IDE')
g=sns.barplot(x='AgeGroup', y='movie_score', data=df1, palette='magma', alpha=0.7,order=sorted(df1['AgeGroup'].unique()),ax=ax[1])
g.set(title='Movie IDE by Age Group', ylim=(0,12), ylabel='')
g=sns.barplot(x='AgeGroup', y='RestMovieDiff', data=df1, palette='magma', alpha=0.7,order=sorted(df1['AgeGroup'].unique()),ax=ax[2])
g.set(title='Rest - Movie IDE by Age Group', ylim=(0,5), ylabel='Difference')
g=sns.barplot(x='AgeGroup', y='ISC', data=df1, palette='magma', alpha=0.7,order=sorted(df1['AgeGroup'].unique()),ax=ax[3])
g.set(title='Movie ISC by Age Group', ylim=(0,.15), ylabel='Correlation')
sns.despine()
plt.tight_layout()

# ISC predicted by Age and movie_FD

In [ ]:
formula = f"ISC ~ Age + sex + session + movie_FD"
temp1 = difference_df.copy()
results = pwr.run_regression_analyses(temp1, 'Age', 'ISC', formula, ['movie_FD','ISC','sex','session', 'Age'], 
                                output_directory=None, root_filename=None, region_order=REGION_ORDER, 
                                title='', plot=0, verbose=1)

In [ ]:
t_mask.max()

In [ ]:
v = results.set_index('region_name').reindex(REGION_ORDER)[f'r2'].values

helper.generate_surface_plot( 
                v,
                image_fn="main_plots/HBN_age_predicts_isc_movieTP_R2_surface.pdf",
                atlas='Schaefer',
                cmap='magma',
                cbar_range=[0, 0.2],
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'Model R^2',
                threshold=None,
                mask_medial_wall=True
            )

to_plot = ['Age']
cmaps = [helper.diverging_colormap_gpu()]
for var,cmap in zip(to_plot, cmaps):
    sig_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'sig_{var}_fdr'].values
    coef_masked = results.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values * sig_mask
    t_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'tstat_{var}'].values * sig_mask

    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, cmap)
    cbar_range = (-0.007, 0.007)
    helper.generate_surface_plot( 
                coef_masked,
                image_fn="main_plots/HBN_age_predicts_isc_movieTP_age_coef_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'$\beta$_{var}',
                threshold=None,
                mask_medial_wall=True
            )
    
    cbar_range, this_cmap = helper.determine_colorbar_range(t_mask, cmap)
    cbar_range=(-9.5, 9.5)
    helper.generate_surface_plot( 
                t_mask,
                image_fn="main_plots/HBN_age_predicts_isc_movieTP_age_tstat_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'tstat_{var}',
                threshold=None,
                mask_medial_wall=True
            )

# Movie ID plots

In [ ]:
formula = f"movie_score ~ Age + ISC + sex + session + movie_FD"
results = pwr.run_regression_analyses(temp1, 'Age', 'movie_score', formula, ['ISC','movie_FD','sex','session', 'Age'], 
                                output_directory=None, root_filename=None, region_order=REGION_ORDER, 
                                title='', plot=0, verbose=1)

In [ ]:
v = results.set_index('region_name').reindex(REGION_ORDER)[f'r2'].values
helper.generate_surface_plot( 
                v,
                image_fn="main_plots/HBN_age_predicts_ide_movieTP_r2_masked.pdf",
                atlas='Schaefer',
                cmap='magma',
                cbar_range=[0,0.6],
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'Model R^2',
                threshold=None,
                mask_medial_wall=True
            )

to_plot = ['Age','ISC']
cmaps = [helper.diverging_colormap_gpu(),helper.diverging_colormap_gpu()]
for var,cmap in zip(to_plot, cmaps):
    sig_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'sig_{var}_fdr'].values
    coef_masked = results.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values * sig_mask
    t_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'tstat_{var}'].values * sig_mask

    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, cmap)
    
    helper.generate_surface_plot( 
                coef_masked,
                image_fn=f"main_plots/HBN_age_predicts_ide_movieTP_{var}_coef_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'$\beta$_{var}',
                threshold=None,
                mask_medial_wall=True
            )
    
    cbar_range, this_cmap = helper.determine_colorbar_range(t_mask, cmap)
    helper.generate_surface_plot( 
                t_mask,
                image_fn=f"main_plots/HBN_age_predicts_ide_movieTP_{var}_tstat_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'tstat_{var}',
                threshold=None,
                mask_medial_wall=True
            )

# Rest ID Plots

In [ ]:
formula = f"rest_score ~ Age + sex + session + rest_FD"
results = pwr.run_regression_analyses(temp1, 'Age', 'rest_score', formula, ['rest_FD','sex','session', 'Age'], 
                                output_directory=None, root_filename=None, region_order=REGION_ORDER, 
                                title='', plot=0, verbose=1)

In [ ]:
v = results.set_index('region_name').reindex(REGION_ORDER)[f'r2'].values
helper.generate_surface_plot( 
                v,
                image_fn="main_plots/HBN_age_predicts_ide_rest_r2_masked.pdf",
                atlas='Schaefer',
                cmap='magma',
                cbar_range=[0,0.5],
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'Model R^2',
                threshold=None,
                mask_medial_wall=True
            )

to_plot = ['Age']
cmaps = [helper.diverging_colormap_gpu()]
for var,cmap in zip(to_plot, cmaps):
    sig_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'sig_{var}_fdr'].values
    coef_masked = results.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values * sig_mask
    t_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'tstat_{var}'].values * sig_mask

    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, cmap)
    
    helper.generate_surface_plot( 
                coef_masked,
                image_fn=f"main_plots/HBN_age_predicts_ide_rest_{var}_coef_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'$\beta$_{var}',
                threshold=None,
                mask_medial_wall=True
            )
    
    cbar_range, this_cmap = helper.determine_colorbar_range(t_mask, cmap)
    helper.generate_surface_plot( 
                t_mask,
                image_fn=f"main_plots/HBN_age_predicts_ide_rest_{var}_tstat_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'tstat_{var}',
                threshold=None,
                mask_medial_wall=True
            )

# Rest - movie ID Difference results

In [ ]:
formula = f"RestMovieDiff ~ Age + ISC + sex + session + movie_FD + rest_FD"
temp1 = difference_df.copy()
results = pwr.run_regression_analyses(temp1, 'Age', 'RestMovieDiff', formula, ['movie_FD','ISC','rest_FD','sex','session', 'Age'], 
                                output_directory=None, root_filename=None, region_order=REGION_ORDER, 
                                title='Rest-Movie ', plot=0, verbose=1)

v = results.set_index('region_name').reindex(REGION_ORDER)[f'r2'].values
# helper.generate_surface_plot( 
#                 v,
#                 image_fn=f"main_plots/HBN_age_predicts_RMDiff_r2_masked.pdf",
#                 atlas='Schaefer',
#                 cmap='magma',
#                 cbar_range=[0,0.3],
#                 surf_type='fslr',
#                 target_density='32k',
#                 method='linear',
#                 include_cbar=True,
#                 title=rf'Model R^2',
#                 threshold=None,
#                 mask_medial_wall=True
#             )

to_plot = ['Age','ISC']
cmaps = [helper.diverging_colormap_gpu(), helper.diverging_colormap_gpu()]
for var,cmap in zip(to_plot, cmaps):
    sig_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'sig_{var}_fdr'].values
    coef_masked = results.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values * sig_mask
    # t_mask = results.set_index('region_name').reindex(REGION_ORDER)[f'tstat_{var}'].values * sig_mask

    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, cmap)
    # Determine if the cbar range contains zero for diverging colormap
    # if cbar_range[0] < 0 < cbar_range[1]:
    #     this_cmap = this_cmap
    # else:
    #     this_cmap = 'magma_r'
    if var == 'ISC': cbar_range = (-21, 21)
    helper.generate_surface_plot( 
                coef_masked,
                image_fn=f"main_plots/HBN_age_predicts_RMDiff_{var}_coef_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'$\beta$_{var}',
                threshold=None,
                mask_medial_wall=True
            )
    
    # cbar_range, this_cmap = helper.determine_colorbar_range(t_mask, cmap)
    # # Determine if the cbar range contains zero for diverging colormap
    # if cbar_range[0] < 0 < cbar_range[1]:
    #     this_cmap = this_cmap
    # else:
    #     this_cmap = 'magma_r'

    # helper.generate_surface_plot( 
    #             t_mask,
    #             image_fn=f"main_plots/HBN_age_predicts_RMDiff_{var}_tstat_masked.pdf",
    #             atlas='Schaefer',
    #             cmap=this_cmap,
    #             cbar_range=cbar_range,
    #             surf_type='fslr',
    #             target_density='32k',
    #             method='linear',
    #             include_cbar=True,
    #             title=rf'tstat_{var}',
    #             threshold=None,
    #             mask_medial_wall=True
    #         )

# ISC by IDE correlation as a function of age

In [ ]:
from numpy import random

In [ ]:
def permute_pattern(data0, data1, n_permutations=1000, random_state=None):
    """Generate spatially permuted versions of a s.

    Args:
        pattern (np.ndarray): 1D array of brain values to permute.
        n_permutations (int): Number of permutations to generate.
        method (str): Method for spatial permutation ('spin' or other).
        random_state (int or None): Random seed for reproducibility.

    Returns:
        np.ndarray: 2D array of shape (n_permutations, len(pattern)) with permuted patterns.
    """
    

    if random_state is not None:
        np.random.seed(random_state)
    permuted_pattern = data0.copy()
    null_correlations = np.zeros(n_permutations)
    for i in range(n_permutations):
        # Generate a random permutation of indices
        perm_indices = np.random.permutation(len(data0))
        permuted_data0 = data0[perm_indices]
        # Compute correlation with data1
        corr, _ = stats.spearmanr(permuted_data0, data1)
        null_correlations[i] = corr
    true_correlation, _ = stats.spearmanr(data0, data1)
    p_value = (np.sum(np.abs(null_correlations) >= np.abs(true_correlation)) + 1) / (n_permutations + 1)
    zscored = (true_correlation - np.mean(null_correlations)) / np.std(null_correlations)
    return p_value, true_correlation, zscored 

In [ ]:
pivot_df = results_df[results_df['measure'].isin(['TPHATE_DiffOp', 'ISC'])].copy()
pivot_df = pivot_df[pivot_df['task'] == 'movieTP']
pivot_df = pivot_df.pivot_table(index=['subject','region_name'], columns='measure', values='score').reset_index()
pivot_df.head()

In [ ]:
#df_corr = pd.DataFrame(columns=['subject','rho','pval','zscore','movie_FD','Age','AgeGroup1','sex','session'])
for s in pivot_df['subject'].unique():
    temp = pivot_df[pivot_df['subject'] == s]
    if len(temp) < 400 or len(par_df[par_df['subject_id'] == s])==0:
        continue
    ide_movie = temp['TPHATE_DiffOp'].values
    isc = temp['ISC'].values
    pval, obs, zsc = stats_helpers.permute_pattern(ide_movie, isc, n_permutations=1000, random_state=None)
    df_corr.loc[len(df_corr)] = [s, obs, pval, zsc, 
                                par_df[par_df['subject_id'] == s]['movie_FD'].values[0],
                                par_df[par_df['subject_id'] == s]['age'].values[0],
                                par_df[par_df['subject_id'] == s]['AgeGroup1'].values[0],
                                par_df[par_df['subject_id'] == s]['sex'].values[0],
                                par_df[par_df['subject_id'] == s]['session'].values[0]
                               ]
    

In [ ]:
df_corr.to_csv('HBN/results/isc_ide_correlation_null_stats.csv')

In [ ]:
sns.barplot(x='AgeGroup1', y='zscore', data=df_corr, palette='magma', alpha=0.6,order=sorted(df_corr['AgeGroup1'].unique()))
sns.stripplot(x='AgeGroup1', y='zscore', data=df_corr, palette='magma', size=4, alpha=1, order=sorted(df_corr['AgeGroup1'].unique()))


In [ ]:
sns.histplot(nulls, kde=True)
# Plot the observed correlation
plt.axvline(obs, color='red', linestyle='--', label='Observed Correlation')
plt.xlabel('Spearman Correlation')
plt.ylabel('Frequency')
plt.title('Null Distribution of Spearman Correlations (Permuted Patterns)')
plt.legend()
plt.show()

In [ ]:
model.params.get('Age', np.nan)

In [ ]:
scipy.stats.spearmanr(df_corr['Age'], df_corr['rho_isc_ide'], nan_policy='omit')

In [ ]:
# correlation plot with line of best fit
sns.regplot(x='Age', y='rho_isc_ide', data=df_corr, scatter_kws={'s': 20, 'alpha':0.5})
# add spearmanr stats to the plot
rho, p = scipy.stats.spearmanr(df_corr['Age'], df_corr['rho_isc_ide'], nan_policy='omit')
ax = plt.gca()
ax.text(0.95, 0.05, f'rho={rho:.3f}\np={p:.3g}\nN={len(df_corr)}',
        transform=ax.transAxes, ha='right', va='bottom', 
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.title('Subject-wise ISC vs IDE (movie) correlation vs Age')
plt.xlabel('Age')
plt.ylabel('Spearman rho (ISC vs IDE_movie)')
sns.despine()

In [ ]:
# Other age grouping
def get_age_groupings2(age):
    if age < 8:
        return 'U08'
    if age < 9:
        return 'U09'
    if age < 10:
        return 'U10'
    if age < 11:
        return 'U11'
    if age < 12:
        return 'U12'
    if age < 13:
        return 'U13'
    if age < 14:
        return 'U14'
    if age < 15:
        return 'U15'
    if age < 16:
        return 'U16'
    if age < 17:
        return 'U17'
    return 'U22'
df_corr['AgeGroup2'] = [get_age_groupings2(a) for a in df_corr['Age']]
df_corr.head()

In [ ]:
age_group_counts

In [ ]:
len(age_group_counts)

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
sns.barplot(x='AgeGroup2', y='rho_isc_ide', data=df_corr, palette='magma', alpha=0.7,order=sorted(df_corr['AgeGroup2'].unique()),ax=ax,edgecolor='black', ci=None)
sns.stripplot(x='AgeGroup2', y='rho_isc_ide', data=df_corr, palette='magma', size=5, alpha=1, order=sorted(df_corr['AgeGroup2'].unique()),ax=ax)
# on each bar, get the nuber of subjects and add to the plot just above the x axis
age_group_counts = df_corr['AgeGroup2'].value_counts().to_dict()
for p,age_group in zip(ax.patches, ['U08','U09','U10','U11','U12','U13','U14','U15','U16','U17','U22']):
    ax.text(p.get_x() + p.get_width() / 2., 0.01,
            f'n={age_group_counts[age_group]}',
            ha="center", va="bottom")
sns.despine()
plt.title('Subject-wise ISC vs IDE (movie) correlation by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Spearman rho (ISC vs IDE_movie)')
plt.axhline(0, color='black', linestyle='--')
plt.ylim(-0.3 , 1)